# Partie 1 — Stockage dans MongoDB
## BigData Commerce Analytics

**Objectifs :**
1. Établir et valider la connexion à l'instance locale MongoDB sur `mongodb://localhost:27017`.
2. Créer et sélectionner la base de données `commerce`.
3. Importer les collections brutes (JSON) et la table de commandes nettoyée (CSV).
4. Vérifier les volumétries et effectuer des requêtes exploratoires.

## Étape 5 — Vérifier la connexion à MongoDB
Depuis Python, établir une connexion à `mongodb://localhost:27017`, vérifier que le serveur répond et sélectionner la base de données `commerce`.

In [1]:
from pymongo import MongoClient

# 1. Connexion au serveur local MongoDB
client = MongoClient("mongodb://localhost:27017/")

# 2. Test du ping pour valider que le serveur répond
try:
    ping_result = client.admin.command('ping')
    print("✅ Connexion réussie à MongoDB ! Réponse :", ping_result)
except Exception as e:
    print("❌ Erreur de connexion :", e)

# 3. Sélection de la base 'commerce'
db = client["commerce"]
print("📁 Base de données active :", db.name)


✅ Connexion réussie à MongoDB ! Réponse : {'ok': 1.0}
📁 Base de données active : commerce


## Étape 6 — Importer les données dans la base `commerce`
Nous insérons les fichiers JSON de `data/raw/` dans leurs collections respectives (`products`, `carts`, `users`, `comments`) ainsi que le fichier nettoyé `commandes_ecommerce_nettoyees.csv` dans la collection `orders`.

In [2]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# Importation des collections JSON brutes
collections_json = ["products", "carts", "users", "comments"]

for name in collections_json:
    json_path = RAW / f"{name}.json"
    if json_path.exists():
        payload = json.loads(json_path.read_text(encoding="utf-8"))
        docs = payload.get(name, [])
        if docs:
            db[name].drop()  # Réinitialise la collection si déjà existante
            result = db[name].insert_many(docs)
            print(f"📦 Collection '{name}' : {len(result.inserted_ids)} documents insérés.")

# Importation de la table de commandes nettoyée (CSV)
csv_clean = PROCESSED / "commandes_ecommerce_nettoyees.csv"
if csv_clean.exists():
    df_orders = pd.read_csv(csv_clean)
    orders_docs = df_orders.to_dict(orient="records")
    db["orders"].drop()
    res_orders = db["orders"].insert_many(orders_docs)
    print(f"📦 Collection 'orders' : {len(res_orders.inserted_ids)} documents insérés.")


📦 Collection 'products' : 194 documents insérés.
📦 Collection 'carts' : 208 documents insérés.
📦 Collection 'users' : 208 documents insérés.
📦 Collection 'comments' : 340 documents insérés.


📦 Collection 'orders' : 788 documents insérés.


## Étape 7 — Vérification des collections et requêtes simples
Affichage des statistiques des collections créées et exemples de requêtes NoSQL.

In [3]:
# Lister les collections disponibles dans la base 'commerce'
print("📁 Collections dans 'commerce' :", db.list_collection_names())

# Compter les documents par collection
stats = []
for col in db.list_collection_names():
    stats.append({"Collection": col, "Nombre_documents": db[col].count_documents({})})

display(pd.DataFrame(stats))


📁 Collections dans 'commerce' : ['comments', 'carts', 'orders', 'products', 'users']


,Collection,Nombre_documents
0,comments,340
1,carts,208
2,orders,788
3,products,194
4,users,208


In [4]:
# Exemple : Afficher un document exemple de la collection 'products'
sample_product = db.products.find_one({}, {"title": 1, "category": 1, "price": 1, "rating": 1, "_id": 0})
print("Exemple de produit :", sample_product)

# Exemple : Trouver tous les produits de la catégorie 'smartphones' ou 'laptops'
tech_products = list(db.products.find(
    {"category": {"$in": ["smartphones", "laptops", "mobile-accessories"]}},
    {"title": 1, "price": 1, "_id": 0}
).limit(5))

display(pd.DataFrame(tech_products))


Exemple de produit : {'title': 'Essence Mascara Lash Princess', 'category': 'beauty', 'price': 9.99, 'rating': 2.56}


,title,price
0,Apple MacBook Pro 14 Inch Space Grey,1999.99
1,Asus Zenbook Pro Dual Screen Laptop,1799.99
2,Huawei Matebook X Pro,1399.99
3,Lenovo Yoga 920,1099.99
4,New DELL XPS 13 9300 Laptop,1499.99
